# Kubernetes 第1周：核心概念

> **学习目标**：搭建本地 K8s 环境，理解 Pod / Deployment / Service 三个核心对象

---

## 从单机到集群：为什么需要 K8s？

你已经会用 Docker 了——一个 `docker run` 就能启动应用。如果只用一台机器，Docker 就够了。

但真实世界是这样的：
- 你有多台服务器（节点），应用要跑在多个节点上（**调度**）
- 某台机器挂了，上面的容器要自动迁移到其他机器（**自愈**）
- 流量大了要自动扩容，流量小了要缩容省成本（**弹性伸缩**）
- 新版本上线要一个一个替换，不能全停（**滚动更新**）
- 服务之间怎么发现对方？怎么负载均衡？（**服务发现**）

**Kubernetes 就是解决这些"多机编排"问题的操作系统。**

```
Docker = 管理单台机器上的容器
K8s    = 管理多台机器上的容器集群
```

类比：
- Docker = 一个工头，管一个车间里的工人（容器）
- K8s = 厂长，管多个车间、多个工头、协调全厂生产

---

## K8s 架构概览

```
                     ┌──────────────────────────┐
                     │     控制平面 (Master)      │
                     │                          │
  kubectl ──────────▶│  API Server  (一切入口)    │
                     │  Scheduler   (调度决策)    │
                     │  Controller  (期望 vs 实际) │
                     │  etcd        (集群记忆)    │
                     └──────────┬───────────────┘
                                │
              ┌─────────────────┼─────────────────┐
              │                 │                 │
     ┌────────┴────────┐ ┌──────┴──────┐ ┌───────┴───────┐
     │  Worker Node 1  │ │ Worker 2   │ │  Worker 3    │
     │  ┌───┐ ┌───┐   │ │ ┌───┐ ┌───┐ │ │ ┌───┐ ┌───┐  │
     │  │ P │ │ P │   │ │ │ P │ │ P │ │ │ │ P │ │ P │  │
     │  └───┘ └───┘   │ │ └───┘ └───┘ │ │ └───┘ └───┘  │
     │  kubelet       │ │  kubelet    │ │  kubelet     │
     └────────────────┘ └─────────────┘ └──────────────┘
```

| 组件 | 在哪 | 职责 |
|------|------|------|
| **API Server** | Master | 一切操作的统一入口，REST API |
| **Scheduler** | Master | 决定 Pod 调度到哪个节点（Filter → Score → Bind） |
| **Controller Manager** | Master | 不断对比"期望状态"和"实际状态"，做调整 |
| **etcd** | Master | 分布式 KV 存储，存集群所有配置和状态 |
| **kubelet** | Worker | 节点上的 Agent，接收指令管理 Pod |
| **kube-proxy** | Worker | 网络代理，实现 Service 的负载均衡 |

### 声明式 vs 命令式

K8s 是**声明式**的：你告诉它"我想要 3 个副本"，它自己想办法达到这个状态。
Docker 是**命令式**的：你告诉它"启动这个容器"、"停掉那个容器"。

类比：
- 命令式 = "把空调调到 26 度"（具体操作）
- 声明式 = "房间应该保持 26 度"（期望状态，恒温器自动调节）

---

## 搭建本地 K8s 环境

学习用不搞集群，两个选择：

| 工具 | 特点 | 推荐场景 |
|------|------|----------|
| **minikube** | 单节点 K8s 在 VM 中 | 功能最全，需要 VM 支持 |
| **kind** | K8s in Docker，每个节点是个容器 | 轻量快速，CI 友好 |

这里用 **kind**（Kubernetes in Docker），前提是已装 Docker。

In [ ]:
# 安装 kind（如果还没装）
# 详见: https://kind.sigs.k8s.io/docs/user/quick-start/

# For Linux:
# ! curl -Lo ./kind https://kind.sigs.k8s.io/dl/v0.20.0/kind-linux-amd64
# ! chmod +x ./kind
# ! sudo mv ./kind /usr/local/bin/kind

# 验证
! kind version 2>/dev/null || echo "kind 未安装，请先安装 kind"

In [ ]:
# 创建 K8s 集群（一个控制平面 + 一个工作节点）
# ! kind create cluster --name k8s-learning

# 如果用 minikube：
# ! minikube start --driver=docker

# 验证集群是否正常
# ! kubectl cluster-info
# ! kubectl get nodes

print("如果已有 K8s 环境，跳过创建步骤，直接验证：")
! kubectl get nodes 2>/dev/null || echo "请先创建 K8s 集群"

---

## kubectl：与 K8s 对话的窗口

`kubectl` 是命令行客户端，你之后 90% 的 K8s 操作都通过它。

### 命令模式

```
kubectl <动词> <资源类型> [资源名] [参数]
```

常用动词：`get` / `describe` / `create` / `apply` / `delete` / `logs` / `exec`

In [ ]:
# 探索集群
! kubectl cluster-info 2>/dev/null
! kubectl version --short 2>/dev/null || kubectl version
! kubectl get nodes 2>/dev/null
! kubectl get namespaces 2>/dev/null

---

## Pod：K8s 的最小调度单元

**Pod 是 K8s 中最小的、最基本的部署单元。** 一个 Pod 包含一个或多个容器。

关键理解：
- Pod 内的容器**共享网络命名空间**（同一个 IP、同一个端口空间）
- Pod 内的容器**可以共享存储卷**
- Pod 是**临时的**——随时可能被销毁和重建

类比：
- Docker 容器 = 一个独立的工人
- Pod = 一个"工位"，上面可能有一个工人（单容器）或多个紧密协作的工人（多容器）

```
Pod: my-pod (IP: 10.244.1.5)
├── Container: app (主容器，跑你的应用)
├── Container: sidecar (边车容器，比如收集日志)
└── Shared: Network / Volume
```

In [ ]:
# 命令式创建 Pod（最简单，但不推荐生产用）
! kubectl run hello-pod --image=python:3.12-slim \
  --restart=Never \
  --command -- python -c "print('Hello from Pod!')" 2>/dev/null

# 查看 Pod 状态
! kubectl get pods 2>/dev/null
! kubectl logs hello-pod 2>/dev/null

In [ ]:
# 声明式：用 YAML 定义 Pod（推荐方式）
! mkdir -p /tmp/k8s-demo

%%writefile /tmp/k8s-demo/pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: my-first-pod
  labels:
    app: demo
spec:
  containers:
  - name: app
    image: python:3.12-slim
    command: ["python", "-c"]
    args:
    - |
      import time
      import socket
      host = socket.gethostname()
      while True:
          print(f"[{host}] 运行中...")
          time.sleep(5)

# 应用 YAML
! kubectl apply -f /tmp/k8s-demo/pod.yaml 2>/dev/null

# 查看
! kubectl get pod my-first-pod -o wide 2>/dev/null
! kubectl describe pod my-first-pod 2>/dev/null | head -30

In [ ]:
# 查看日志（-f 实时跟踪，类似 tail -f）
! kubectl logs my-first-pod 2>/dev/null

# 进入 Pod 执行命令
! kubectl exec my-first-pod -- python -c "import socket; print('主机名:', socket.gethostname())" 2>/dev/null

In [ ]:
# 清理
! kubectl delete pod hello-pod --wait=false 2>/dev/null
! kubectl delete pod my-first-pod --wait=false 2>/dev/null

---

## Deployment：管理 Pod 的生命周期

你不会直接创建 Pod（太脆弱，挂了就没了）。**Deployment** 帮你管理 Pod：

- 保证指定数量的副本在运行
- 滚动更新（一个一个替换，不停机）
- 回滚（版本出问题了可以退回去）

```
Deployment
  └── ReplicaSet (v1) ─── 管理 Pod 的版本
  │     ├── Pod-abc
  │     ├── Pod-def
  │     └── Pod-ghi
  └── ReplicaSet (v2) ─── 滚动更新时创建的新 RS
        ├── Pod-jkl
        ├── Pod-mno
        └── Pod-pqr
```

**你操作 Deployment，Deployment 操作 ReplicaSet，ReplicaSet 操作 Pod。** 这是一条"控制链"。

In [ ]:
%%writefile /tmp/k8s-demo/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: py-api
spec:
  replicas: 3                    # 我要 3 个副本
  selector:
    matchLabels:
      app: py-api                # 管理带这个标签的 Pod
  template:                      # Pod 模板
    metadata:
      labels:
        app: py-api
    spec:
      containers:
      - name: api
        image: python:3.12-slim
        command: ["python", "-c"]
        args:
        - |
          from http.server import HTTPServer, BaseHTTPRequestHandler
          import socket, os
          host = socket.gethostname()
          class H(BaseHTTPRequestHandler):
              def do_GET(self):
                  self.send_response(200)
                  self.end_headers()
                  self.wfile.write(f"Hello from {host}! v1\n".encode())
          HTTPServer(("0.0.0.0", 8000), H).serve_forever()
        ports:
        - containerPort: 8000

! kubectl apply -f /tmp/k8s-demo/deployment.yaml 2>/dev/null
! kubectl get deploy,rs,pods 2>/dev/null

In [ ]:
# 演示滚动更新：修改镜像版本
! kubectl set image deployment/py-api api=python:3.12-slim 2>/dev/null
! kubectl rollout status deployment/py-api 2>/dev/null

# 查看滚动更新历史
! kubectl rollout history deployment/py-api 2>/dev/null

In [ ]:
# 扩缩容
! kubectl scale deployment py-api --replicas=5 2>/dev/null
! sleep 2
! kubectl get pods -l app=py-api 2>/dev/null

# 缩回去
! kubectl scale deployment py-api --replicas=2 2>/dev/null
! sleep 2
! kubectl get pods -l app=py-api 2>/dev/null

---

## Service：让 Pod 可被访问

Pod 的 IP 是临时的（Pod 重建后 IP 就变了），而且 Deployment 有多个副本——**你该访问哪个 IP？**

**Service** 提供一个**固定的虚拟 IP（ClusterIP）**和 DNS 名称，后端自动负载均衡到健康的 Pod。

```
客户端 → Service (ClusterIP: 10.96.1.50, 稳定不变)
               ↓  通过 label selector 找到 Pod
          ┌────┴────┬────────┐
        Pod A     Pod B    Pod C
      (10.244.1.5) (10.244.2.3) (10.244.1.8)
```

### Service 类型

| 类型 | 说明 | 使用场景 |
|------|------|----------|
| **ClusterIP** | 集群内部 IP（默认） | 服务间互访 |
| **NodePort** | 在每个节点上开一个端口（30000-32767） | 开发调试、临时暴露 |
| **LoadBalancer** | 云厂商的负载均衡器 | 生产对外暴露 |
| **ExternalName** | DNS 别名，指向外部服务 | 外部服务映射 |

In [ ]:
%%writefile /tmp/k8s-demo/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: py-api-svc
spec:
  selector:
    app: py-api            # 选择带这个标签的 Pod
  ports:
  - port: 80               # Service 的端口
    targetPort: 8000       # Pod 的端口
  type: ClusterIP          # 仅集群内部访问

! kubectl apply -f /tmp/k8s-demo/service.yaml 2>/dev/null
! kubectl get svc py-api-svc 2>/dev/null

In [ ]:
# 验证 Service 是否正常工作
# 方法1：通过 Service 的 ClusterIP 访问（需要在集群内部执行）
! kubectl run test-curl --image=curlimages/curl --rm -it --restart=Never \
  -- curl -s py-api-svc 2>/dev/null || echo "(kind 网络可能需要调整)"

# 方法2：端口转发到本地（调试最常用！）
print("在另一个终端执行：kubectl port-forward service/py-api-svc 8080:80")
print("然后浏览器打开 http://localhost:8080")

# 方法3：查看 Service 的 Endpoints（确认后端 Pod 已就绪）
! kubectl get endpoints py-api-svc 2>/dev/null

---

## ConfigMap & Secret：配置与敏感信息

**不要把配置写死在镜像里！** 同一个镜像应该在开发、测试、生产都能用，区别只是配置不同。

- **ConfigMap**：非敏感配置（数据库 URL、日志级别、应用参数）
- **Secret**：敏感信息（密码、Token、证书）

In [ ]:
# ---- ConfigMap ----
%%writefile /tmp/k8s-demo/configmap.yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: app-config
data:
  DEBUG: "false"
  LOG_LEVEL: "info"
  DATABASE_HOST: "postgres-svc"     # 服务名就是 DNS
  DATABASE_PORT: "5432"

# ---- Secret ----
%%writefile /tmp/k8s-demo/secret.yaml
apiVersion: v1
kind: Secret
metadata:
  name: app-secret
type: Opaque
stringData:              # stringData：明文字段（K8s 自动编码为 base64）
  DATABASE_PASSWORD: "my-secret-password"
  API_KEY: "sk-1234567890abcdef"

! kubectl apply -f /tmp/k8s-demo/configmap.yaml 2>/dev/null
! kubectl apply -f /tmp/k8s-demo/secret.yaml 2>/dev/null

In [ ]:
# 查看创建的 ConfigMap 和 Secret
! kubectl get configmap app-config -o yaml 2>/dev/null | head -15
print("...")
! kubectl get secret app-secret 2>/dev/null

# 注意：Secret 的值在 etcd 中默认只是 base64 编码，不是加密！
! kubectl get secret app-secret -o jsonpath="{.data.DATABASE_PASSWORD}" 2>/dev/null | base64 -d

In [ ]:
# 演示：在 Pod 中使用 ConfigMap 和 Secret
%%writefile /tmp/k8s-demo/pod-with-config.yaml
apiVersion: v1
kind: Pod
metadata:
  name: config-demo
spec:
  containers:
  - name: app
    image: python:3.12-slim
    command: ["python", "-c"]
    args:
    - |
      import os
      # 环境变量方式（来自 ConfigMap）
      print("DEBUG:", os.environ.get("DEBUG"))
      print("LOG_LEVEL:", os.environ.get("LOG_LEVEL"))
      # 环境变量方式（来自 Secret）
      print("DB_PASSWORD:", os.environ.get("DB_PASSWORD")[:4] + "***")
      # 文件挂载方式
      for f in os.listdir("/etc/config"):
          with open(f"/etc/config/{f}") as fp:
              print(f"config file {f}: {fp.read().strip()}")
    env:
    - name: DEBUG
      valueFrom:
        configMapKeyRef:
          name: app-config
          key: DEBUG
    - name: LOG_LEVEL
      valueFrom:
        configMapKeyRef:
          name: app-config
          key: LOG_LEVEL
    - name: DB_PASSWORD
      valueFrom:
        secretKeyRef:
          name: app-secret
          key: DATABASE_PASSWORD
    volumeMounts:
    - name: config-volume
      mountPath: /etc/config
  volumes:
  - name: config-volume
    configMap:
      name: app-config
  restartPolicy: Never

! kubectl apply -f /tmp/k8s-demo/pod-with-config.yaml 2>/dev/null
! sleep 3
! kubectl logs config-demo 2>/dev/null

In [ ]:
# 清理本课资源
! kubectl delete deployment py-api --wait=false 2>/dev/null
! kubectl delete service py-api-svc --wait=false 2>/dev/null
! kubectl delete pod config-demo --wait=false 2>/dev/null
! kubectl delete pod hello-pod --wait=false 2>/dev/null
! kubectl delete configmap app-config --wait=false 2>/dev/null
! kubectl delete secret app-secret --wait=false 2>/dev/null
print("清理完成")

---

## Namespace、Label、Selector

### Namespace：资源的"文件夹"

K8s 用 Namespace 隔离资源。默认有四个：
- `default`：你创建资源如果不指定 ns，就在这里
- `kube-system`：K8s 系统组件
- `kube-public`：公开的配置数据
- `kube-node-lease`：节点心跳数据

### Label & Selector：资源的"标签"和"筛选器"

K8s 中几乎所有东西都用 label 来组织和关联：
- Service 通过 label selector 找到 Pod
- Deployment 通过 label selector 管理 Pod
- 你通过 label 筛选 `kubectl get pods -l app=py-api`

In [ ]:
# Namespace 操作演示
! kubectl create namespace demo-ns 2>/dev/null || echo "命名空间已存在"
! kubectl get namespaces 2>/dev/null

# 在指定 namespace 中创建资源
! kubectl run test-pod --image=busybox --restart=Never -n demo-ns \
  --command -- sleep 5 2>/dev/null
! kubectl get pods -n demo-ns 2>/dev/null

# 用 label 筛选
! kubectl get pods -n demo-ns --show-labels 2>/dev/null

In [ ]:
# 清理
! kubectl delete namespace demo-ns --wait=false 2>/dev/null

---

## 🎯 第1周总结

| 概念 | 一句话 | 类比 |
|------|--------|------|
| **Pod** | 最小调度单元，1+ 容器共享网络/存储 | 一个"工位" |
| **Deployment** | 管理 Pod 副本、滚动更新、回滚 | 生产线主管 |
| **Service** | 稳定入口 + 负载均衡 | 公司总机转接 |
| **ConfigMap** | 非敏感配置 | 设置文件 |
| **Secret** | 密码/Token/证书 (base64) | 保险柜 |
| **Namespace** | 资源逻辑隔离 | 文件夹 |
| **Label** | 给资源打标签，用于筛选和关联 | 便利贴 |

### Deployment + Service 关系图

```
                     Service (py-api-svc)
                     ClusterIP: 10.96.1.50
                           │
              ┌────────────┼────────────┐
              ▼            ▼            ▼
         Pod (app=py-api) Pod          Pod
         10.244.1.5      10.244.2.3   10.244.1.8
              ▲            ▲            ▲
              └────────────┴────────────┘
                     Deployment (py-api)
                     replicas: 3
```

---

## 🧪 综合练习：将 Compose 项目迁移到 K8s

把你 Docker Week 3 做的 Flask + Redis 计数器项目用 K8s 资源重新部署：

要求：
1. 创建 `my-app` Namespace
2. Web 用 Deployment（2 副本）
3. Redis 用 Deployment（1 副本）
4. 两个 Service（Web: ClusterIP, 端口 5000）
5. 用 ConfigMap 传配置（Redis 地址）
6. 验证：`kubectl exec` 进 Web Pod，curl Redis Service

In [ ]:
# 你的练习代码
pass